# 日本株セクター週足レジーム検知 — 手法比較分析

日足セクターデータから週足特徴量を作成し、複数のレジーム検知アルゴリズムを
ウォークフォワードで適用・スコアリングします。

**データソース**:
- `synthetic`: 合成データ（デモ）
- `project`: 本リポジトリ `data/features/`（J-Quants 配置済みの場合）
- ファイルパス: CSV/Parquet（`date, sector, close` 列）

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from invest_system.research.sector_regime import (
    PipelineConfig,
    load_sector_daily,
    build_weekly_features,
    run_pipeline,
    SCORE_WEIGHTS,
)
from invest_system.research.sector_regime.evaluation import selection_rationale
from invest_system.research.sector_regime.visualization import (
    plot_regime_on_price,
    plot_method_comparison_heatmap,
    plot_score_breakdown,
)

import matplotlib.pyplot as plt
%matplotlib inline

## 1. データ読み込みと週足特徴量

In [ ]:
DATA_SOURCE = "synthetic"  # "project" または Parquet/CSV パスに変更
SECTORS = None  # 例: ["3300", "5250", "7050"]

if DATA_SOURCE == "synthetic":
    from invest_system.research.sector_regime.data_loader import generate_synthetic_sector_daily
    daily = generate_synthetic_sector_daily(sectors=SECTORS)
else:
    daily = load_sector_daily(DATA_SOURCE)
    if SECTORS:
        daily = daily[daily["sector"].isin(SECTORS)]

weekly = build_weekly_features(daily)
print(f"セクター数: {daily['sector'].nunique()}")
print(f"日足期間: {daily['date'].min()} 〜 {daily['date'].max()}")
weekly.head()

## 2. 全手法の比較・スコアリング

In [ ]:
config = PipelineConfig(warmup_weeks=104, min_weeks=260)
print("スコア重み:", SCORE_WEIGHTS)

result = run_pipeline(
    DATA_SOURCE,
    sectors=SECTORS,
    config=config,
    output_dir=None,
    show_progress=True,
)

result.ranking.head(15)

## 3. セクター別ベスト手法

In [ ]:
best = result.best
best[["sector", "method", "composite_score", "pred_score", "quality_score", "stability_score", "econ_score"]]

In [ ]:
for _, row in best.iterrows():
    print(f"\n【{row['sector']}】 {row['method']}")
    print(selection_rationale(row))

## 4. 可視化

In [ ]:
plot_method_comparison_heatmap(result.ranking)
plt.show()

plot_score_breakdown(result.best)
plt.show()

In [ ]:
# 代表セクターのレジーム色分けチャート
for sector in best["sector"].head(3):
    method = best.loc[best["sector"] == sector, "method"].iloc[0]
    out = next(o for o in result.outputs[sector] if o.method == method)
    plot_regime_on_price(result.weekly, out, title=f"{sector} — {method}")
    plt.show()

## 5. セクター特性とベスト手法の考察

In [ ]:
print(result.sector_insight)